# تحلیل اکتشافی داده های اضطراب اجتماعی

**فاز دوم: EDA**

اعضای تیم: علی خوش اخلاق، محمدحسین میرمعصومی، آرمین نورمحمدی، علی کریمی، محسن منصف

## 1. کتابخانه ها و تنظیمات

از پایتون 3.13 استفاده کنید و ورژن های زیر

In [17]:
# pandas==3.0.5 numpy==2.2.6 plotly==7.1.0 scipy==1.16.3 statsmodels==0.15.0

In [1]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import scipy
import statsmodels
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pio.renderers.default = "notebook"
px.defaults.template = "plotly_white"
px.defaults.height = 450
BLUE = "#4C78A8"
RED = "#E45756"
TARGET = "Anxiety Level (1-10)"
LABEL = "Target"

## 2. داده های اولیه


In [2]:
df = pd.read_csv("social_anxiety_dataset.csv")
print(df.shape)
df.head()

(2030, 22)


,Age,Gender,Occupation,Sleep Hours,Physical Activity (hrs/week),Caffeine Intake (mg/day),Alcohol Consumption (drinks/week),Smoking,Family History of Anxiety,Stress Level (1-10),Heart Rate (bpm),Breathing Rate (breaths/min),Sweating Level (1-5),Dizziness,Medication,Therapy Sessions (per month),Recent Major Life Event,Diet Quality (1-10),Anxiety Level (1-10),Target,is_Anxious,Therapy History
0,59.0,Other,Teacher,7.0,2.4,40.0,5,Yes,No,4,79,21,5,No,No,0,Yes,1.0,2.0,0,0,Group Therapy
1,46.0,Female,Student,5.1,5.4,156.0,11,NaN,No,3,70,19,4,Yes,No,2,No,1.0,4.0,0,0,NaN
2,40.0,Other,Lawyer,5.1,1.9,570.0,14,Yes,Yes,9,101,23,3,No,No,6,Yes,4.0,9.0,1,1,NaN
3,40.0,Male,Nurse,7.6,0.9,129.0,0,No,No,9,116,16,2,No,NaN,2,No,2.0,6.0,0,0,NaN
4,26.0,Male,Other,6.7,3.0,64.0,13,No,No,15,119,19,4,No,Yes,0,Yes,4.0,3.0,0,0,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2030 entries, 0 to 2029
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                1968 non-null   float64
 1   Gender                             1911 non-null   str    
 2   Occupation                         2030 non-null   str    
 3   Sleep Hours                        1994 non-null   float64
 4   Physical Activity (hrs/week)       2030 non-null   float64
 5   Caffeine Intake (mg/day)           1936 non-null   float64
 6   Alcohol Consumption (drinks/week)  2030 non-null   int64  
 7   Smoking                            1924 non-null   str    
 8   Family History of Anxiety          2030 non-null   str    
 9   Stress Level (1-10)                2030 non-null   int64  
 10  Heart Rate (bpm)                   2030 non-null   int64  
 11  Breathing Rate (breaths/min)       2030 non-null   int64  
 12  Swe

In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Age,1968.0,NaN,NaN,NaN,39.921748,13.243754,18.0,29.0,40.0,51.0,64.0
Gender,1911,3,Female,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Occupation,2030,13,Student,182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sleep Hours,1994.0,NaN,NaN,NaN,6.424624,2.161544,-10.0,5.8,6.7,7.5,11.0
Physical Activity (hrs/week),2030.0,NaN,NaN,NaN,2.800246,2.233899,-10.0,1.4,2.8,4.2,9.2
Caffeine Intake (mg/day),1936.0,NaN,NaN,NaN,306.275826,208.012717,0.0,177.75,277.5,391.0,1500.0
Alcohol Consumption (drinks/week),2030.0,NaN,NaN,NaN,9.490148,6.028653,-10.0,5.0,10.0,15.0,19.0
Smoking,1924,2,Yes,1020,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Family History of Anxiety,2030,2,Yes,1058,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Stress Level (1-10),2030.0,NaN,NaN,NaN,6.028571,3.137269,1.0,3.0,6.0,9.0,15.0


In [5]:
pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Missing_%': (df.isna().mean() * 100).round(1),
    'unique': df.nunique(),
})

,Missing,Missing_%,unique
Age,62,3.1,47
Gender,119,5.9,3
Occupation,0,0.0,13
Sleep Hours,36,1.8,76
Physical Activity (hrs/week),0,0.0,89
Caffeine Intake (mg/day),94,4.6,544
Alcohol Consumption (drinks/week),0,0.0,23
Smoking,106,5.2,2
Family History of Anxiety,0,0.0,2
Stress Level (1-10),0,0.0,11


In [7]:
print(df.duplicated().sum())

0


##  مشاهدات اولیه

- سطر هامون 2030 تاس و ستون هامون 22 تاس
- 9 تا ستون با داده گمشده داریم
 - که Therapy History با 90 درصد داده گمشده بدترینه
- Sleep Hours، Physical Activity و Alcohol مقدار منفی دارند
- Stress Level مقدار 15 دارد که منطقی نیست
- دوتا ستون Target و is_Anxious تغریبا یکسانند
- ستون Heart Rate و Caffeine مقدار خارج از بازه دارند

## 3. تحلیل و پاک سازی داده

### 3.1 جمعیت شناختی: Age, Gender, Occupation

### 3.2 سبک زندگی: Sleep Hours, Physical Activity, Caffeine Intake, Alcohol Consumption, Smoking, Diet Quality

### 3.3 فیزیولوژیک: Heart Rate, Breathing Rate, Sweating Level, Dizziness

### 3.4 درمان، سابقه و متغیر هدف: Family History, Medication, Therapy Sessions, Therapy History, Recent Major Life Event, Stress Level, Anxiety Level, Target

### 3.5 ادغام پاک سازی ها و ساخت clean_df

## 4. ویژوال تک متغیره

### 4.1 جمعیت شناختی

### 4.2 سبک زندگی

### 4.3 فیزیولوژیک

### 4.4 درمان، سابقه و متغیر هدف

## 5. ویژوال دومتغیره

### 5.1 ستون های دسته ای با اضطراب

### 5.2 ستون های عددی با اضطراب

## 6. آزمون های آماری

### 6.1 آزمون همبستگی: عددی با عددی

### 6.2 آزمون t-test و ANOVA: دسته ای با عددی

### 6.3 آزمون chi-square: دسته ای با دسته ای

### 6.4 امتیازی: بازه های اطمینان

## 7. KPI و فیچرهای تعاملی

### 7.1 استخراج KPI و فیچرهای تعاملی از ستون ها

### 7.2 ویژوال KPI و فیچرهای جدید

## 8. ویژوال چندمتغیره: ترکیب ویژگی ها و گروه های پرخطر

## 9. امتیازی: ویژوال سه متغیره و بیشتر